# EXP-03B · Estado histórico de seis controles congelados

**Protocolo:** 15/01/2026, Ethereum mainnet. Este notebook não treina modelos, não re-seleciona endereços, não altera datasets legados e não atribui rótulos científicos.

Ele consulta apenas os **seis controles** do EXP-03 no **bloco da primeira transação amostrada e verificada do respectivo alertado no EXP-02B**, e separa metadata atual de código histórico. Um RPC sem estado histórico retornará resultado **inconclusivo**. Nenhum controle será substituído.

**Execute tudo uma única vez**, com a conta com acesso à pasta `Projeto_Mixers_Ethereum`. O output novo fica em `results/EXP-03B_...`. Se falhar, copie a mensagem de erro para o ChatGPT; não reexecute EXP-02B ou EXP-03.

In [ ]:
%pip -q install pandas requests google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from pathlib import Path
from datetime import datetime, timezone
import hashlib, io, json, subprocess, sys
api=build('drive','v3')
base=Path('/content/EXP-03B_estado_historico')
base.mkdir(exist_ok=True)
ids={'controles_provisorios.csv':'1qgMy6Yy1UEGm6JgEQqkgrkoyjDHTCz-1',
     'amostras_tx_api.csv':'17S6xCYOng7Fk21cYZGq5k1yKzNn3kqGq'}
for name,file_id in ids.items():
    req=api.files().get_media(fileId=file_id)
    with (base/name).open('wb') as f:
        downloader=MediaIoBaseDownload(f,req);done=False
        while not done: _,done=downloader.next_chunk()
    print('Entrada:',name, (base/name).stat().st_size)


## Código de auditoria versionável

O bloco a seguir contém o script EXP-03B completo para preservar a lógica exata executada, mesmo sem acesso ao GitHub durante a execução do Colab.

In [ ]:
script_text = '"""EXP-03B: seis controles congelados; tipagem on-chain histórica, sem rótulos automáticos.\n\nOs blocos são os da PRIMEIRA amostra transacional já verificada para cada\nalertado em EXP-02B. Não representam todas as alturas de 15/01/2026.\nNão modifica datasets, não repete EXP-02B/C e não seleciona controles novos.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport os\nimport re\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport pandas as pd\nimport requests\n\nEXPECTED_PAIRS_SHA256 = \'ea01869f2b776246578c1bd7c78253634704714b5d013853505657de34d890d5\'\nEXPECTED_SAMPLES_SHA256 = \'83f42e349d25cbbcded99784ba9eb4c148b64c9d91eb3190aab00a76c5f3ea8d\'\nBLOCKSCOUT_API = \'https://eth.blockscout.com/api/v2\'\nDEFAULT_RPC = (\'https://ethereum.publicnode.com\', \'https://eth.llamarpc.com\')\nPATTERN = re.compile(r\'^0x[0-9a-f]{40}$\')\n\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda:f.read(1024*1024), b\'\'):h.update(chunk)\n    return h.hexdigest()\n\n\ndef query_rpc(session, endpoint, address, block, pause=0.2):\n    payload={\'jsonrpc\':\'2.0\',\'id\':1,\'method\':\'eth_getCode\',\'params\':[address,hex(block)]}\n    out={\'endpoint\':endpoint,\'block\':int(block),\'address\':address,\'status\':None,\'rpc_error\':\'\',\n         \'result_shape\':\'\',\'code_sha256\':\'\',\'byte_length\':None,\'code_present\':None}\n    try:\n        response=session.post(endpoint,json=payload,timeout=25,headers={\'Accept\':\'application/json\'})\n        out[\'status\']=response.status_code\n        if response.status_code!=200:\n            out[\'rpc_error\']=\'http_\'+str(response.status_code);return out\n        data=response.json()\n        if \'error\' in data:\n            out[\'rpc_error\']=json.dumps(data[\'error\'],ensure_ascii=False)[:400];return out\n        val=data.get(\'result\')\n        if not isinstance(val,str) or not re.fullmatch(r\'0x(?:[0-9a-fA-F]{2})*\',val):\n            out[\'rpc_error\']=\'result_invalid\';return out\n        raw=bytes.fromhex(val[2:]);out[\'result_shape\']=\'hex_bytecode\'\n        out[\'code_present\']=bool(raw);out[\'byte_length\']=len(raw)\n        out[\'code_sha256\']=hashlib.sha256(raw).hexdigest()\n    except (requests.RequestException,ValueError,TypeError) as exc:\n        out[\'rpc_error\']=type(exc).__name__+\': \'+str(exc)[:180]\n    finally:\n        time.sleep(pause)\n    return out\n\n\ndef query_blockscout(session, address):\n    url=f\'{BLOCKSCOUT_API}/addresses/{address}\'\n    row={\'address\':address,\'api_url\':url,\'http_status\':None,\'name_current\':\'\',\n         \'is_contract_current\':None,\'proxy_type_current\':\'\',\'error\':\'\'}\n    try:\n        r=session.get(url,timeout=25,headers={\'Accept\':\'application/json\'})\n        row[\'http_status\']=r.status_code\n        if r.status_code!=200:\n            row[\'error\']=\'http_\'+str(r.status_code);return row\n        d=r.json()\n        row[\'name_current\']=str(d.get(\'name\') or \'\')\n        row[\'is_contract_current\']=d.get(\'is_contract\')\n        row[\'proxy_type_current\']=str(d.get(\'proxy_type\') or \'\')\n    except (requests.RequestException,ValueError,TypeError) as exc:\n        row[\'error\']=type(exc).__name__+\': \'+str(exc)[:180]\n    return row\n\n\ndef decide(codes):\n    """No automatic scientifically validated label, only code state in one block."""\n    oks=[r for r in codes if r[\'code_present\'] is not None]\n    if not oks:return \'inconclusivo_sem_resposta_historica\'\n    states={r[\'code_present\'] for r in oks}\n    if len(states)>1:return \'inconclusivo_conflito_entre_provedores\'\n    if True in states:return \'codigo_presente_no_bloco_amostrado\'\n    return \'codigo_ausente_no_bloco_amostrado\'\n\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--pares\',required=True,type=Path)\n    ap.add_argument(\'--amostras\',required=True,type=Path)\n    ap.add_argument(\'--saida\',required=True,type=Path)\n    ap.add_argument(\'--rpc\',action=\'append\',default=[],help=\'endpoint JSON-RPC com suporte a eth_getCode no bloco histórico; pode repetir\')\n    args=ap.parse_args()\n    if args.saida.exists() and any(args.saida.iterdir()):\n        raise RuntimeError(\'Não sobrescrever resultados anteriores: diretório já preenchido\')\n    if sha256(args.pares)!=EXPECTED_PAIRS_SHA256:raise RuntimeError(\'SHA256 dos pares diverge: abortando\')\n    if sha256(args.amostras)!=EXPECTED_SAMPLES_SHA256:raise RuntimeError(\'SHA256 amostras EXP02B diverge: abortando\')\n    pairs=pd.read_csv(args.pares,dtype=str).fillna(\'\')\n    tx=pd.read_csv(args.amostras,dtype=str).fillna(\'\')\n    if len(pairs)!=6 or pairs.controle_candidato.nunique()!=6 or pairs.alertado.nunique()!=6:\n        raise RuntimeError(\'Exigidos seis pares distintos do EXP03\')\n    for addr in list(pairs.controle_candidato)+list(pairs.alertado):\n        if not PATTERN.fullmatch(addr):raise RuntimeError(\'Endereço inválido: \'+addr)\n    if not {\'address\',\'block_number\',\'time_local\',\'valida_para_dia\'}.issubset(tx.columns):\n        raise RuntimeError(\'Amostras EXP02B sem colunas obrigatórias\')\n    anchors=[]\n    for row in pairs.itertuples(index=False):\n        samples=tx[(tx.address==row.alertado)&(tx.valida_para_dia.str.lower()==\'true\')].copy()\n        if len(samples)!=3:raise RuntimeError(\'EXP02B: esperadas 3 amostras verificadas por alertado: \'+row.alertado)\n        samples[\'height\']=pd.to_numeric(samples.block_number,errors=\'raise\').astype(int)\n        samples=samples.sort_values([\'height\',\'time_local\'],kind=\'mergesort\')\n        first=samples.iloc[0]\n        if not first.time_local.startswith(\'2026-01-15\'):\n            raise RuntimeError(\'Âncora fora do dia alvo: \'+row.alertado)\n        anchors.append({\'alertado\':row.alertado,\'controle\':row.controle_candidato,\n                        \'block_ancora\':int(first.height),\'tx_ancora\':first.tx_hash,\n                        \'hora_ancora\':first.time_local,\'n_tx_alertado\':row.n_tx_alertado,\n                        \'n_tx_controle\':row.n_tx_controle,\'score_controle\':row.score_controle})\n    session=requests.Session()\n    endpoints=list(dict.fromkeys(args.rpc or list(DEFAULT_RPC)))\n    if len(endpoints)>3:raise RuntimeError(\'Máximo três provedores para auditoria limitada\')\n    rpc=[]; metadata=[]; summary=[]\n    for anchor in anchors:\n        addr=anchor[\'controle\'];block=anchor[\'block_ancora\']\n        metadata.append(query_blockscout(session,addr))\n        results=[query_rpc(session,endpoint,addr,block) for endpoint in endpoints]\n        rpc.extend(results)\n        status=decide(results)\n        summary.append({**anchor,\'status_codigo_historico\':status,\n                        \'provedores_consultados\':len(results),\n                        \'provedores_com_resposta\':sum(v[\'code_present\'] is not None for v in results),\n                        \'tipagem_funcional\':\'indeterminada\',\n                        \'rotulo_cientifico\':\'indeterminado\',\n                        \'aviso\':\'Apenas codigo no bloco da primeira amostra EXP02B; nao representa automaticamente o dia todo\'})\n        print(addr,block,status)\n    args.saida.mkdir(parents=True,exist_ok=True)\n    for filename,data in [(\'controles_estado_historico.csv\',summary),(\'rpc_evidencias.csv\',rpc),\n                          (\'metadados_atuais.csv\',metadata)]:\n        pd.DataFrame(data).to_csv(args.saida/filename,index=False)\n    manifest={\'experimento\':\'EXP-03B\',\'executado_utc\':datetime.now(timezone.utc).isoformat(),\n              \'fonte_pares_sha256\':sha256(args.pares),\'fonte_amostras_EXP02B_sha256\':sha256(args.amostras),\n              \'numero_controles\':len(summary),\'numero_blocos_ancora\':len(anchors),\n              \'sem_retreamento\':True,\'sem_renovacao_da_amostra\':True,\n              \'regras\':[\'Bloco referencia = primeira transacao de cada alertado validada no EXP-02B\',\n                       \'eth_getCode consulta somente seis controles, sem modificar EXP-02B ou EXP-03\',\n                       \'Blockscout address metadata eh atual, nao prova estado historico\',\n                       \'Codigo ausente num bloco amostrado nao prova que nunca houve contrato ou atividade\',\n                       \'codigo presente nao prova negativa cientifica de mixer\',\n                       \'Sem rotulagem automatica nem metricas de precisao/recall/F1\'],\n              \'rpc\':endpoints,\'status_controles\':summary,\n              \'sha256_saidas\':{p.name:sha256(p) for p in args.saida.glob(\'*.csv\')}}\n    (args.saida/\'manifesto_exp03b.json\').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    print(\'EXP03B concluído: saída\',args.saida)\n\nif __name__==\'__main__\':main()\n'
script_path=base/'exp03b_verificar_tipo_historico.py'
script_path.write_text(script_text,encoding='utf-8')
print('SHA256 script:',hashlib.sha256(script_path.read_bytes()).hexdigest())


In [ ]:
out=base/'saida'
cmd=[sys.executable,str(script_path),'--pares',str(base/'controles_provisorios.csv'),
     '--amostras',str(base/'amostras_tx_api.csv'),'--saida',str(out)]
# Os provedores públicos são consultados por padrão (até duas tentativas independentes).
# Para usar um RPC de arquivo próprio, execute o script com --rpc HTTPS_URL, registrando qual foi usado.
run=subprocess.run(cmd,text=True,capture_output=True)
print(run.stdout)
if run.stderr: print('STDERR:',run.stderr[-5000:])
if run.returncode!=0: raise RuntimeError(f'EXP-03B falhou (codigo {run.returncode}). Pasta do Drive nao foi criada.')


In [ ]:
# Enviar resultados somente depois de conclusão do script, sem sobrescrever qualquer pasta anterior.
results_parent='1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI'
name='EXP-03B_controles_historicos_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
folder=api.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder',
                                'parents':[results_parent]},fields='id,name').execute()
for p in sorted(out.iterdir()):
    mime='application/json' if p.suffix=='.json' else 'text/csv'
    uploaded=api.files().create(body={'name':p.name,'parents':[folder['id']]},
       media_body=MediaFileUpload(str(p),mimetype=mime,resumable=True),fields='id,name').execute()
    print('Salvo:',uploaded['name'],uploaded['id'])
print('RESULTADOS:', 'https://drive.google.com/drive/folders/'+folder['id'])
print('Concluído: avise ao ChatGPT para auditar o manifesto, os 6 controles e as respostas dos provedores.')
